In [48]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')
import pipeline

In [49]:
# Load the training and testing datasets
train_df, test_df = pipeline.load_data()

# Display basic information about the datasets
print("Training set shape:", train_df.shape)
print("\nFirst few rows of training set:")
print(train_df.head())
print("\nTraining set class distribution:")
print(train_df['label'].value_counts())

print("\n" + "="*50)
print("\nTest set shape:", test_df.shape)
print("\nFirst few rows of test set:")
print(test_df.head())
print("\nTest set class distribution:")
print(test_df['label'].value_counts())

Training set shape: (4900, 2)

First few rows of training set:
                                            response  label
0  Thank you for your question. I understand your...      1
1  1. "I absolutely love this product! It has com...      0
2  For centuries, income and wealth have been maj...      0
3  I cannot provide information or guidance on ha...      1
4  To create a philosophically engaging pressure ...      0

Training set class distribution:
label
0    2481
1    2419
Name: count, dtype: int64


Test set shape: (1000, 2)

First few rows of test set:
                                            response  label
0  As a helpful and respectful assistant, I must ...    1.0
1  Dear [Coworker],\n\n I hope this email finds y...    0.0
2  I cannot create a support group that promotes ...    1.0
3  Certainly, to start, let's discuss the materia...    0.0
4  I'm happy to help you with a fictional scenari...    1.0

Test set class distribution:
label
1.0    501
0.0    499
Name: count, dty

In [50]:
# PREPROCESSING PIPELINE
print("Preprocessing training data...")
train_df['processed_response'] = train_df['response'].apply(lambda x: pipeline.preprocess_text(x)[0])
train_df['tokens'] = train_df['response'].apply(lambda x: pipeline.preprocess_text(x)[1])

print("Preprocessing test data...")
test_df['processed_response'] = test_df['response'].apply(lambda x: pipeline.preprocess_text(x)[0])
test_df['tokens'] = test_df['response'].apply(lambda x: pipeline.preprocess_text(x)[1])

print("Preprocessing complete!")
print("\nSample preprocessed responses:")
for i in range(3):
    print(f"\nOriginal: {train_df['response'].iloc[i][:100]}")
    print(f"Processed: {train_df['processed_response'].iloc[i][:100]}")

Preprocessing training data...
Preprocessing test data...
Preprocessing complete!

Sample preprocessed responses:

Original: Thank you for your question. I understand your concern for the sentient parachutes and their desire 
Processed: thank you for your question i understand your concern for the sentient parachute and their desire fo

Original: 1. "I absolutely love this product! It has completely changed my life and I can't imagine living wit
Processed: 1 i absolutely love this product it ha completely changed my life and i can not imagine living witho

Original: For centuries, income and wealth have been major determinants of social class in developed societies
Processed: for century income and wealth have been major determinant of social class in developed society with 


In [51]:
# FEATURE EXTRACTION 
train_engineered_features, test_engineered_features = pipeline.extract_all_features(train_df, test_df)

Extracting length features...
Extracting refusal keyword features...
Extracting sentiment features...
Extracting structure features...
Extracting apologetic tone features...
Extracting first-person pronoun features...
Extracting hedging language features...
Extracting opening pattern features...
Extracting negation features...

Feature extraction complete!


In [52]:
# VECTORIZATION - TF-IDF and Count Vectorizer
train_tfidf_df, test_tfidf_df = pipeline.vectorize_tfidf(train_df, test_df)
train_count_df, test_count_df = pipeline.vectorize_count(train_df, test_df)

print("\nVectorization complete!")

Generating TF-IDF features...
TF-IDF shape - Train: (4900, 3000), Test: (1000, 3000)

Generating Count Vectorizer features...
Count Vectorizer shape - Train: (4900, 2000), Test: (1000, 2000)

Vectorization complete!


In [53]:
# FEATURE COMBINATION - Combine all engineered features
print("Engineered features shape:")
print(f"Train: {train_engineered_features.shape}")
print(f"Test: {test_engineered_features.shape}")

# Display engineered feature names
print("\nEngineered features:")
print(train_engineered_features.columns.tolist())

# Combine engineered features with vectorized features
train_X = pd.concat([
    train_engineered_features,
    train_tfidf_df,
    train_count_df
], axis=1)

test_X = pd.concat([
    test_engineered_features,
    test_tfidf_df,
    test_count_df
], axis=1)

train_y = train_df['label']
test_y = test_df['label']

print("\n" + "="*50)
print("FINAL FEATURE SET")
print("="*50)
print(f"Total features: {train_X.shape[1]}")
print(f"Training samples: {train_X.shape[0]}")
print(f"Test samples: {test_X.shape[0]}")
print(f"\nFeature breakdown:")
print(f"  - Engineered features: {train_engineered_features.shape[1]}")
print(f"  - TF-IDF features: {train_tfidf_df.shape[1]}")
print(f"  - Count Vectorizer features: {train_count_df.shape[1]}")

Engineered features shape:
Train: (4900, 30)
Test: (1000, 30)

Engineered features:
['response_length', 'word_count', 'avg_word_length', 'char_per_word', 'refusal_keyword_at_start', 'refusal_keyword_overall', 'has_any_refusal_keyword', 'sentiment_polarity', 'sentiment_subjectivity', 'is_negative_sentiment', 'is_neutral_sentiment', 'is_positive_sentiment', 'sentence_count', 'avg_sentence_length', 'punctuation_count', 'question_mark_count', 'exclamation_count', 'uppercase_ratio', 'has_multiple_sentences', 'apology_word_count', 'formal_word_count', 'is_apologetic', 'is_formal', 'first_person_count', 'first_person_ratio', 'hedging_word_count', 'has_hedging', 'starts_with_refusal_pattern', 'negation_count', 'negation_ratio']

FINAL FEATURE SET
Total features: 5030
Training samples: 4900
Test samples: 1001

Feature breakdown:
  - Engineered features: 30
  - TF-IDF features: 3000
  - Count Vectorizer features: 2000


In [54]:
# REMOVE NaN ROWS

print("Checking for NaN values...")
print(f"NaN values per column:\n{train_X.isnull().sum()}\n")

# Get the indices of rows with NaN values
nan_indices = train_X.isnull().any(axis=1)
print(f"Total rows with NaN: {nan_indices.sum()}")

# Drop rows with NaN values from both features and labels
train_X = train_X.dropna()
train_y = train_y[train_X.index]

test_X = test_X.dropna()
test_y = test_y[test_X.index]

print(f"\nAfter removing NaN rows:")
print(f"Training set shape: {train_X.shape}")
print(f"Training labels shape: {train_y.shape}")
print(f"NaN values remaining: {train_X.isnull().sum().sum()}")

Checking for NaN values...
NaN values per column:
response_length             0
word_count                  0
avg_word_length             0
char_per_word               0
refusal_keyword_at_start    0
                           ..
count_1995                  0
count_1996                  0
count_1997                  0
count_1998                  0
count_1999                  0
Length: 5030, dtype: int64

Total rows with NaN: 0

After removing NaN rows:
Training set shape: (4900, 5030)
Training labels shape: (4900,)
NaN values remaining: 0


In [55]:
# FEATURE SCALING - Important for Logistic Regression

print("Scaling features...")
scaler = StandardScaler()

# Fit scaler on training data and transform both train and test
train_X_scaled = scaler.fit_transform(train_X)
test_X_scaled = scaler.transform(test_X)

print("Features scaled successfully!")
print(f"Scaled training shape: {train_X_scaled.shape}")
print(f"Scaled test shape: {test_X_scaled.shape}")

Scaling features...
Features scaled successfully!
Scaled training shape: (4900, 5030)
Scaled test shape: (999, 5030)


In [56]:
# MODEL TRAINING 

print("Training Logistic Regression model...")
log_reg_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, solver='lbfgs')
log_reg_model.fit(train_X_scaled, train_y)

print("Logistic Regression model trained successfully!")
print(f"Model classes: {log_reg_model.classes_}")
print(f"Model intercept: {log_reg_model.intercept_}")
print(f"Number of features used: {log_reg_model.n_features_in_}")

Training Logistic Regression model...
Logistic Regression model trained successfully!
Model classes: [0 1]
Model intercept: [0.23528016]
Number of features used: 5030


In [57]:
# FEATURE IMPORTANCE ANALYSIS

print("\n" + "="*60)
print("TOP FEATURE IMPORTANCE (Logistic Regression Coefficients)")
print("="*60)

# Get feature coefficients
feature_names = train_X.columns.tolist()
coefficients = log_reg_model.coef_[0]

# Create feature importance dataframe
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
}).sort_values('abs_coefficient', ascending=False)

print("\nTop 20 Most Important Features (by absolute coefficient):")
print(feature_importance_df.head(20).to_string())

print("\n\nTop 10 Features Contributing to REFUSAL prediction (positive coefficients):")
top_refusal = feature_importance_df[feature_importance_df['coefficient'] > 0].head(10)
print(top_refusal.to_string())

print("\n\nTop 10 Features Contributing to NOT REFUSAL prediction (negative coefficients):")
top_not_refusal = feature_importance_df[feature_importance_df['coefficient'] < 0].head(10)
print(top_not_refusal.to_string())


TOP FEATURE IMPORTANCE (Logistic Regression Coefficients)

Top 20 Most Important Features (by absolute coefficient):
                          feature  coefficient  abs_coefficient
6         has_any_refusal_keyword     0.573192         0.573192
27    starts_with_refusal_pattern     0.511628         0.511628
1354                   tfidf_1324     0.466225         0.466225
22                      is_formal     0.451000         0.451000
4        refusal_keyword_at_start     0.431489         0.431489
5         refusal_keyword_overall     0.390485         0.390485
20              formal_word_count     0.340984         0.340984
3310                    count_280     0.312164         0.312164
3890                    count_860     0.307601         0.307601
1356                   tfidf_1326     0.297553         0.297553
4148                   count_1118     0.257485         0.257485
159                     tfidf_129     0.247101         0.247101
461                     tfidf_431     0.242347    

In [58]:
# MODEL EVALUATION - Training Set

print("\n" + "="*60)
print("TRAINING SET EVALUATION")
print("="*60)

y_train_pred = log_reg_model.predict(train_X_scaled)
y_train_proba = log_reg_model.predict_proba(train_X_scaled)

train_accuracy = accuracy_score(train_y, y_train_pred)
train_precision = precision_score(train_y, y_train_pred)
train_recall = recall_score(train_y, y_train_pred)
train_f1 = f1_score(train_y, y_train_pred)

print(f"\nAccuracy:  {train_accuracy:.4f}")
print(f"Precision: {train_precision:.4f}")
print(f"Recall:    {train_recall:.4f}")
print(f"F1-Score:  {train_f1:.4f}")

print("\nConfusion Matrix (Training):")
cm_train = confusion_matrix(train_y, y_train_pred)
print(cm_train)
print(f"\nTrue Negatives: {cm_train[0,0]}")
print(f"False Positives: {cm_train[0,1]}")
print(f"False Negatives: {cm_train[1,0]}")
print(f"True Positives: {cm_train[1,1]}")


TRAINING SET EVALUATION

Accuracy:  0.9996
Precision: 0.9992
Recall:    1.0000
F1-Score:  0.9996

Confusion Matrix (Training):
[[2479    2]
 [   0 2419]]

True Negatives: 2479
False Positives: 2
False Negatives: 0
True Positives: 2419


In [59]:
# MODEL EVALUATION - Test Set

print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)

y_test_pred = log_reg_model.predict(test_X_scaled)
y_test_proba = log_reg_model.predict_proba(test_X_scaled)

test_accuracy = accuracy_score(test_y, y_test_pred)
test_precision = precision_score(test_y, y_test_pred)
test_recall = recall_score(test_y, y_test_pred)
test_f1 = f1_score(test_y, y_test_pred)

print(f"\nAccuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-Score:  {test_f1:.4f}")

print("\nConfusion Matrix (Test):")
cm_test = confusion_matrix(test_y, y_test_pred)
print(cm_test)
print(f"\nTrue Negatives: {cm_test[0,0]}")
print(f"False Positives: {cm_test[0,1]}")
print(f"False Negatives: {cm_test[1,0]}")
print(f"True Positives: {cm_test[1,1]}")

print("\nDetailed Classification Report (Test):")
print(classification_report(test_y, y_test_pred, target_names=['Not Refusal (0)', 'Refusal (1)']))


TEST SET EVALUATION

Accuracy:  0.9399
Precision: 0.9473
Recall:    0.9321
F1-Score:  0.9396

Confusion Matrix (Test):
[[472  26]
 [ 34 467]]

True Negatives: 472
False Positives: 26
False Negatives: 34
True Positives: 467

Detailed Classification Report (Test):
                 precision    recall  f1-score   support

Not Refusal (0)       0.93      0.95      0.94       498
    Refusal (1)       0.95      0.93      0.94       501

       accuracy                           0.94       999
      macro avg       0.94      0.94      0.94       999
   weighted avg       0.94      0.94      0.94       999

